# 01 — Data preparation (all free sources)
Builds: contested-topic file, World-Bank ground truths + **headroom screen**, IPIP check,
and the **cited BFI-2 norms** for the human-calibrated condition (required before full runs).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
print("project root:", ROOT)

project root: /home/raad/Papers/socialLLMs/traitmix


In [2]:
# 1) Topics: starter set + (recommended) Chuang et al. statements + ANES-derived items.
import json
topics = [
 {"id": "T_guncontrol", "statement": "Stricter national gun-control laws would make society safer overall."},
 {"id": "T_immigration", "statement": "Current levels of immigration benefit the country more than they cost it."},
 {"id": "T_neutral_filler", "statement": "Pineapple belongs on pizza.", "role": "filler"},
]
# ANES 2024 (free registration): https://electionstudies.org/data-center/2024-time-series-study/
# -> Optionally rephrase 2-4 policy items as statements and append here, citing variable IDs.
# Chuang et al. materials: https://github.com/yunshiuan/llm-agent-opinion-dynamics
(ROOT / "data" / "topics").mkdir(parents=True, exist_ok=True)
(ROOT / "data" / "topics" / "topics.json").write_text(json.dumps(topics, indent=1))
print("topics.json written:", [t["id"] for t in topics])

topics.json written: ['T_guncontrol', 'T_immigration', 'T_neutral_filler']


In [3]:
# 2) World Bank ground truths (free API; graceful manual fallback)
from traitmix.data import load_ci_estimation
items, truths = load_ci_estimation(require_truths=True)
truths

{'wb_ken_pop': 55339003.0,
 'wb_ind_internet': 55.9,
 'wb_vnm_gdp': 433805036898.465,
 'wb_deu_life': 80.6080487804878}

In [4]:
# 3) HEADROOM SCREEN (needs vLLM up): keep items where the solo model's median relative
# error >= 15% -> writes the filtered wb_items.json used by all experiments.
RUN_SCREEN = False   # set True on your GPU machine
if RUN_SCREEN:
    import json, re, numpy as np
    from traitmix.llm import VLLMClient
    from traitmix.data import DEFAULT_WB_ITEMS
    llm = VLLMClient(model="meta-llama/Llama-3.1-8B-Instruct")
    keep = []
    for it in DEFAULT_WB_ITEMS:
        outs = llm.generate_batch([("You are a careful estimator.",
                f"[ESTIMATE] {it['question']} Reply with a single number only.")]*20, max_tokens=12, temperature=0.8)
        vals = [float(m.group()) for o in outs if (m := re.search(r"-?\d[\d,]*\.?\d*(?:[eE][+-]?\d+)?", (o or "").replace(",", "")))]
        err = np.median([abs(v - truths[it["id"]]) / truths[it["id"]] for v in vals]) if vals else 1.0
        print(it["id"], "median rel err:", round(float(err), 3), "->", "KEEP" if err >= 0.15 else "DROP")
        if err >= 0.15: keep.append(it)
    (ROOT / "data" / "ci" / "wb_items.json").write_text(json.dumps(keep, indent=1))
    print("kept", len(keep), "items")

In [5]:
# 4) IPIP-NEO-120: verify the real item file is in place (public domain: https://ipip.ori.org)
from traitmix.data import load_ipip
try:
    items, name = load_ipip(); print(name, len(items), "items OK")
except FileNotFoundError as e:
    print("ACTION NEEDED:\n", e)

ACTION NEEDED:
 /home/raad/Papers/socialLLMs/traitmix/data/ipip/ipip_neo_120.csv not found. Download the public-domain IPIP-NEO-120 items from https://ipip.ori.org (or J.A. Johnson's IPIP-NEO materials), save as CSV with columns item_text,trait,keyed. Demo battery is only allowed for smoke tests (demo_ok=True).


In [6]:
# 5) BFI-2 human norms (REQUIRED before full runs): paste published normative
# means/SDs (rescaled to [0,1]) and the 5x5 trait correlation matrix, WITH the citation.
NORMS = {
  "citation": "TODO e.g. Soto & John (2017), J. Pers. Soc. Psychol., Table X",
  "mu":    {"openness": None, "conscientiousness": None, "extraversion": None, "agreeableness": None, "neuroticism": None},
  "sigma": {"openness": None, "conscientiousness": None, "extraversion": None, "agreeableness": None, "neuroticism": None},
  "corr":  None,  # 5x5 nested list, trait order O,C,E,A,N
}
import yaml, glob
if all(v is not None for v in NORMS["mu"].values()) and NORMS["corr"] and "TODO" not in NORMS["citation"]:
    (ROOT / "configs" / "norms.yaml").write_text(yaml.safe_dump(NORMS))
    for f in glob.glob(str(ROOT / "configs" / "e2" / "e2_human*.yaml")):
        cfg = yaml.safe_load(open(f)); cfg["composition"] = {"mu": NORMS["mu"], "sigma": NORMS["sigma"], "corr": NORMS["corr"]}
        open(f, "w").write(yaml.safe_dump(cfg, sort_keys=False))
    print("human-calibrated configs patched with cited norms.")
else:
    print("Fill NORMS with published values + citation, then re-run this cell. Full runs are blocked until then.")

Fill NORMS with published values + citation, then re-run this cell. Full runs are blocked until then.
